# Swapped judge check: train on NUDT, test on official (and the reverse)
Same features, GBM and frozen decision rule as T7. Only the training photos change. Train set = 150 photos of one set (our dumps), so the judge is much smaller than Palash's (5 sets). Cut: Learn-then-Test on out-of-fold P of the training set, floor 0.9 as in T7.

In [ ]:
import os, sys, json
os.environ.setdefault("OMP_NUM_THREADS", "6")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import joblib, numpy as np, pandas as pd
from train.checks import replay, query_policy
from train.checks.decision_signals import build_signal_table
from train.checks.candidate_judge import add_context, add_features, baseline, decide_frozen, judge_new, summary, paired_gain
T7 = ROOT / "tests/Test_Iteration_7/results"
DUMPS = T7 / "rich_dump"
SETS = {"official150": "official150", "nudt707": "nudt_fresh707"}
frozen = json.load(open(T7 / "frozen_rules_t7.json")); model = joblib.load(T7 / "judge_t7.joblib")
photos = {s: replay.load_set(DUMPS / f) for s, f in SETS.items()}
print({s: len(p) for s, p in photos.items()})

In [ ]:
rows = []
for s, ph in photos.items():
    for pid, photo in ph.items():
        r = replay.decide_current(photo); rows.append(dict(data=s, photo=pid, stage=r["stage"], pts=int(r["points"]), err_sigma=r["err_sigma"], n_queries=r["n_queries"], inliers=r["inliers"]))
        v = query_policy.run_policy(photo, z_veto=(1.3, 15))
        rows.append(dict(data=s + "|veto", photo=pid, stage=v["stage"], pts=int(v["points"]) if v["answered"] else 0, err_sigma=v["err_sigma"], n_queries=v["n_queries"], inliers=np.nan))
ours_all = pd.DataFrame(rows)
ours = ours_all[~ours_all.data.str.contains("veto")]
veto = ours_all[ours_all.data.str.contains("veto")].assign(data=lambda d: d.data.str.replace("|veto", "", regex=False))
print(summary(ours)); print(summary(veto))

In [ ]:
tabs = {}
for s, f in SETS.items():
    t = build_signal_table(DUMPS / f, s, 10)
    t.to_csv(T7 / f"signals_ours_{s}.csv.gz", index=False)
    tabs[s] = add_context(add_features(t)).reset_index(drop=True)
table = pd.concat(tabs.values(), ignore_index=True)
p = None
print({s: (len(t), t.photo.nunique()) for s, t in tabs.items()})

In [ ]:
from train.checks.candidate_judge import freeze, FROZEN_VARIANTS, FROZEN
import warnings; warnings.filterwarnings("ignore")
out = {"ours (agree+escal)": ours}
for tr, te in [("nudt707","official150"),("official150","nudt707")]:
    try:
        m, tau = freeze(table, [tr], "pc")
    except Exception as e:
        print("freeze failed", repr(e)); m = None
        from train.checks.candidate_judge import judge_probability
        _, m = judge_probability(table, [tr], FROZEN["features"], FROZEN["kind"]); tau = FROZEN["tau_floor"]
    print(f"train {tr} -> test {te}: tau={tau}")
    p = judge_new(m, table)
    for v in ["pc","pc_early"]:
        r = decide_frozen(table, p, tau, v)
        out[f"judge trained on {tr} [{v}]"] = r
rows=[]
for k,v in out.items():
    s = summary(v).reset_index(); s["rule"]=k; rows.append(s)
tab = pd.concat(rows)[["rule","data","score","wrong","answered","queries","n"]]
# keep only held-out rows for swapped judges
keep = ~tab.rule.str.startswith("judge") | ((tab.rule.str.contains("nudt707")&(tab.data=="official150"))|(tab.rule.str.contains("trained on official150")&(tab.data=="nudt707")))
tab[keep].round(2)

In [ ]:
rows=[]
for k in out:
    if not k.startswith("judge"): continue
    te = "official150" if "trained on nudt707" in k else "nudt707"
    sub = lambda d: d[d.data==te]
    g = paired_gain(sub(ours), sub(out[k]), [te]); rows.append(dict(rule=k, test=te, **g))
pd.DataFrame(rows)

In [ ]:
tab[keep].to_csv(T7/"ours_check_swapped_train.csv", index=False)